# MCLDNN — 4-Class Ablation Training (BPSK Removed)
**Dataset**: RML2016.10a · Classes: QPSK, 8PSK, QAM16, QAM64 *(BPSK excluded)*  
**Model**: MCLDNN (TF2 / Keras 3) — identical architecture, output = 4  
**Purpose**: Ablation study — compare against 5-class baseline to measure
the effect of including BPSK on classification of the remaining 4 classes.

### Kaggle Inputs Required
- Dataset: `rml2016-4class` → contains `RML2016.10a_4class.pkl`
- Dataset: `amr-repo` OR use git clone below


In [ ]:
# ── CELL 1: Environment Setup ────────────────────────────────────────────────
import subprocess, sys

GITHUB_REPO = 'https://github.com/YOUR_USERNAME/AMR.git'  # <-- update this
BRANCH      = 'main'
REPO_DIR    = '/kaggle/working/AMR'

!git clone --branch {BRANCH} --depth 1 {GITHUB_REPO} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
%cd {REPO_DIR}
!pip install -q pyyaml

import tensorflow as tf, keras
print(f'TF={tf.__version__}  Keras={keras.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── CELL 2: Configure experiment ─────────────────────────────────────────────
import yaml, os

CONFIG_PATH = 'configs/exp_4class_ablation.yaml'

with open(CONFIG_PATH, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['dataset']['path'] = '/kaggle/input/rml2016-4class/RML2016.10a_4class.pkl'

MOD_CONFIG = '/kaggle/working/exp_4class_ablation_kaggle.yaml'
with open(MOD_CONFIG, 'w') as f:
    yaml.dump(cfg, f)

print('Config:')
print(yaml.dump(cfg, default_flow_style=False))

In [ ]:
# ── CELL 3: (Optional) Resume from checkpoint ─────────────────────────────────
RESUME_WEIGHTS = None
# RESUME_WEIGHTS = '/kaggle/input/rml2016-checkpoints/4class_ablation_epoch90.weights.h5'

In [ ]:
# ── CELL 4: Run Training ─────────────────────────────────────────────────────
resume_flag = f'--resume {RESUME_WEIGHTS}' if RESUME_WEIGHTS else ''
!python src/train.py --config {MOD_CONFIG} {resume_flag}

In [ ]:
# ── CELL 5: Inspect results ───────────────────────────────────────────────────
import os, csv, pickle, numpy as np
from IPython.display import Image, display
import glob

EXP_DIR = 'experiments/4class_ablation'

# Test score
score_file = os.path.join(EXP_DIR, 'results', 'test_score.csv')
if os.path.exists(score_file):
    with open(score_file) as f:
        print('Test Score:')
        for row in csv.reader(f): print(' ', row)

# Per-SNR accuracy
acc_path = os.path.join(EXP_DIR, 'results', 'acc.dat')
if os.path.exists(acc_path):
    acc = pickle.load(open(acc_path,'rb'))
    print(f'\nPeak accuracy: {max(acc.values()):.4f} at SNR={max(acc,key=acc.get)} dB')

# Figures
for fig in sorted(glob.glob(f'{EXP_DIR}/figures/*.png'))[:6]:
    print(fig); display(Image(fig))

In [ ]:
# ── CELL 6: Record run metadata ───────────────────────────────────────────────
import subprocess
commit = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
print(f'Git commit: {commit}')
print('\n>> Upload experiments/4class_ablation/checkpoints/best_model.weights.h5')
print('   to Kaggle Dataset "rml2016-checkpoints" or GitHub Release.')